In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features/all_image_paths.pkl
/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features/all_perceptual_hashes.pkl
/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features/all_cnn_embeddings.h5
/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features/all_hist_features.pkl
/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features/all_sift_features.pkl
/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features/all_orb_features.pkl
/kaggle/input/datasets/mahnoormughal16/pdc-repo/PDC_distributed-reverse-image-search/.gitignore
/kaggle/input/datasets/mahnoormughal16/pdc-repo/PDC_distributed-reverse-image-search/README.md
/kaggle/input/datasets/mahnoormughal16/pdc-repo/PDC_distributed-reverse-image-search/validation/run_evaluation.py
/kaggle/input/datasets/mahnoormughal16/pdc-repo/PDC_distributed-reverse-image-search/validation/evaluation_results.csv
/kaggle/input/datasets/mahnoormughal16/pdc-repo/PDC_distributed-r

In [4]:
import os

# Define the base paths based on the actual Kaggle structure
REPO_BASE = '/kaggle/input/datasets/mahnoormughal16/pdc-repo/PDC_distributed-reverse-image-search'
FEATURES_BASE = '/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features'

# Verify the Repo structure
print("Contents of Repo:")
print(os.listdir(REPO_BASE))

# Verify the Features structure
print("\nContents of Features Dataset:")
print(os.listdir(FEATURES_BASE))

Contents of Repo:
['validation', '.gitignore', 'feature_Fusion', 'README.md', 'query_engine', 'lsh_index', '.git', 'data', 'feature_Extraction']

Contents of Features Dataset:
['all_image_paths.pkl', 'all_perceptual_hashes.pkl', 'all_cnn_embeddings.h5', 'all_hist_features.pkl', 'all_sift_features.pkl', 'all_orb_features.pkl']


In [5]:
import os, sys

# ── Repo code path ──────────────────────────────────────────────────────────
REPO_BASE     = '/kaggle/input/datasets/mahnoormughal16/pdc-repo/PDC_distributed-reverse-image-search'
FEATURES_BASE = '/kaggle/input/datasets/piranchalghai/pdc-micflickr-1m-features'

# ── Add repo to Python path so imports work ─────────────────────────────────
sys.path.insert(0, REPO_BASE)

# ── All feature file paths ───────────────────────────────────────────────────
CNN_PATH   = os.path.join(FEATURES_BASE, 'all_cnn_embeddings.h5')
HASH_PATH  = os.path.join(FEATURES_BASE, 'all_perceptual_hashes.pkl')
SIFT_PATH  = os.path.join(FEATURES_BASE, 'all_sift_features.pkl')
ORB_PATH   = os.path.join(FEATURES_BASE, 'all_orb_features.pkl')
HIST_PATH  = os.path.join(FEATURES_BASE, 'all_hist_features.pkl')
PATHS_PATH = os.path.join(FEATURES_BASE, 'all_image_paths.pkl')

# ── Index paths (inside the repo) ────────────────────────────────────────────
INDEX_DIR  = os.path.join(REPO_BASE, 'lsh_index')

# ── Confirm all files exist ──────────────────────────────────────────────────
files_to_check = {
    'CNN embeddings' : CNN_PATH,
    'Hashes'         : HASH_PATH,
    'SIFT'           : SIFT_PATH,
    'ORB'            : ORB_PATH,
    'Histograms'     : HIST_PATH,
    'Image paths'    : PATHS_PATH,
    'lsh_config.json': os.path.join(INDEX_DIR, 'lsh_config.json'),
    'projection_matrices': os.path.join(INDEX_DIR, 'projection_matrices.pkl'),
    'table_0'        : os.path.join(INDEX_DIR, 'hash_tables', 'table_0.pkl'),
    'table_9'        : os.path.join(INDEX_DIR, 'hash_tables', 'table_9.pkl'),
}

print("File check:")
all_ok = True
for name, path in files_to_check.items():
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "MISSING"
    status = "✓" if exists else "✗"
    print(f"  {status} {name:25s} {size}")
    if not exists:
        all_ok = False

print("\n✓ All files found — ready to proceed." if all_ok else "\n✗ Fix missing files before continuing.")

File check:
  ✓ CNN embeddings            481.8 MB
  ✓ Hashes                    54.8 MB
  ✓ SIFT                      545.4 MB
  ✓ ORB                       161.4 MB
  ✓ Histograms                417.4 MB
  ✓ Image paths               84.7 MB
  ✓ lsh_config.json           0.0 MB
  ✓ projection_matrices       0.1 MB
  ✓ table_0                   4.9 MB
  ✓ table_9                   4.9 MB

✓ All files found — ready to proceed.


In [6]:
from lsh_index.lsh_structure import LSHConfig, hash_embedding
from feature_Fusion.feature_fusion import All_Features, rank_candidates
from query_engine.query_processor import QueryProcessor
from query_engine.search_engine import SearchEngine

print("✓ All imports successful")

✓ All imports successful


In [7]:
qp = QueryProcessor(
    index_dir = INDEX_DIR,
    data_path = CNN_PATH,
)
print("✓ QueryProcessor ready")

Config loaded: 10 tables, hash_size=12, max_candidates=5000
Projection matrices loaded: 10 × (12, 128)
Loading 10 hash tables from /kaggle/input/datasets/mahnoormughal16/pdc-repo/PDC_distributed-reverse-image-search/lsh_index/hash_tables ...
  table_0: 4,096 buckets, 1,000,000 image ID entries
  table_1: 4,096 buckets, 1,000,000 image ID entries
  table_2: 4,096 buckets, 1,000,000 image ID entries
  table_3: 4,096 buckets, 1,000,000 image ID entries
  table_4: 4,096 buckets, 1,000,000 image ID entries
  table_5: 4,096 buckets, 1,000,000 image ID entries
  table_6: 4,096 buckets, 1,000,000 image ID entries
  table_7: 4,096 buckets, 1,000,000 image ID entries
  table_8: 4,096 buckets, 1,000,000 image ID entries
  table_9: 4,096 buckets, 1,000,000 image ID entries
All tables loaded in 1.5s

Building id→index mapping from HDF5 ...
  Mapping built: 1,000,000 entries

✓ QueryProcessor ready


In [8]:
import random

test_ids = random.sample(list(qp.id_to_index.keys()), 10)

print(f"Testing {len(test_ids)} random queries...\n")
for image_id in test_ids:
    emb        = qp.prepare_query_embedding(image_id=image_id)
    candidates = qp.get_candidates(emb)
    print(f"  ID {image_id:>7d} → {len(candidates):>5d} candidates")

print("\nQuery stats:")
stats = qp.get_query_stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

Testing 10 random queries...

  ID   94205 →  3289 candidates
  ID  483416 →  4263 candidates
  ID  150597 →  5000 candidates
  ID  623692 →  2492 candidates
  ID  179102 →  4143 candidates
  ID  144166 →  5000 candidates
  ID  295419 →  2470 candidates
  ID  406459 →  4310 candidates
  ID  231462 →  4770 candidates
  ID  911444 →  5000 candidates

Query stats:
  total_queries: 10
  mean_latency_ms: 3.199934959411621
  p50_latency_ms: 3.123760223388672
  p95_latency_ms: 3.9779424667358394
  p99_latency_ms: 4.0941572189331055


In [9]:
import sys, os

# Patch feature_fusion.py's hardcoded paths by passing ours explicitly
# We instantiate All_Features directly with our paths
feature_store = All_Features(
    cnn_path   = CNN_PATH,
    hash_path  = HASH_PATH,
    sift_path  = SIFT_PATH,
    orb_path   = ORB_PATH,
    hist_path  = HIST_PATH,
    paths_path = PATHS_PATH,
)
print("✓ All_Features loaded")

Loading All_Features ...
  Loading CNN embeddings ...
  ✓ CNN: 1,000,000 embeddings, dim=128
  Loading perceptual hashes ...
  ✓ Hashes: 1,000,000 entries
  Loading SIFT features ...
  ✓ SIFT: 1,000,000 entries
  Loading ORB features ...
  ✓ ORB: 1,000,000 entries
  Loading histograms ...
  ✓ Histograms: 1,000,000 entries
  Loading image paths ...
  ✓ Image paths: 1,000,000 entries
✓ All_Features ready.

✓ All_Features loaded


In [12]:
# Check Ground Truth ID range
gt_ids = [int(k) for k in ground_truth.keys()]
print(f"Ground Truth ID Range: {min(gt_ids)} to {max(gt_ids)}")

# Check Dataset ID range
dataset_ids = list(qp.id_to_index.keys())
print(f"Loaded Dataset ID Range: {min(dataset_ids)} to {max(dataset_ids)}")

Ground Truth ID Range: 2000000 to 2000594
Loaded Dataset ID Range: 0 to 999999


In [16]:
import h5py
import json
import os
import numpy as np

# ── 1. Load Ground Truth and Test Embeddings ────────────────────────────────
GT_PATH = os.path.join(REPO_BASE, 'validation', 'ground_truth.json')
TEST_CNN_PATH = os.path.join(REPO_BASE, 'validation', 'test_features/test_cnn.h5')

with open(GT_PATH) as f:
    ground_truth = json.load(f)

with h5py.File(TEST_CNN_PATH, 'r') as f:
    test_embeddings = f['embeddings'][:]
    test_ids = f['image_ids'][:]

test_id_to_idx = {int(gid): idx for idx, gid in enumerate(test_ids)}
test_query_ids = [int(k) for k in list(ground_truth.keys())[:10]]

print(f"Running validation for {len(test_query_ids)} test queries (IDs 2,000,000+)...\n")

hits      = 0
total_var = 0

# ── 2. Run the Validation Loop ──────────────────────────────────────────────
for query_id in test_query_ids:
    idx = test_id_to_idx[query_id]
    raw_emb = test_embeddings[idx]

    # FIX: QueryProcessor.prepare_query_embedding should handle normalization 
    emb = qp.prepare_query_embedding(raw_embedding=raw_emb)

    # Task 3: Retrieve candidates from the 1M index [cite: 210, 227]
    candidates = qp.get_candidates(emb)

    # Task 4: Manual Re-ranking with proper normalization [cite: 273]
    scores = []
    for cid in list(candidates):
        cand_emb = feature_store.get_cnn(cid).astype(np.float32)
        
        # CRITICAL FIX: Manually normalize candidate before dot product
        norm = np.linalg.norm(cand_emb)
        if norm > 1e-8:
            cand_emb = cand_emb / norm
            
        # Now score will be between -1.0 and 1.0
        score = np.dot(emb, cand_emb)
        scores.append((cid, score))
    
    results = sorted(scores, key=lambda x: x[1], reverse=True)[:10]

    # ── 3. Evaluate Metrics ──────────────────────────────────────────────────
    result_ids = [r[0] for r in results]
    variants   = [int(v) for v in ground_truth.get(str(query_id), [])]
    
    # Check if any variants were even in the candidate set to begin with
    found      = [v for v in variants if v in result_ids]
    
    hits      += len(found)
    total_var += len(variants)

    print(f"  Query {query_id:>7d} | candidates={len(candidates):>5d} | "
          f"variants found: {len(found)}/{len(variants)} | "
          f"top result: ID={results[0][0]} score={results[0][1]:.4f}")

precision = hits / (len(test_query_ids) * 10) if test_query_ids else 0
recall    = hits / total_var if total_var else 0
print(f"\nPrecision@10 : {precision:.3f}  (target > 0.5)")
print(f"Recall@10    : {recall:.3f}  (target > 0.6)")

Running validation for 10 test queries (IDs 2,000,000+)...

  Query 2000000 | candidates= 2657 | variants found: 0/5 | top result: ID=851 score=0.9980
  Query 2000006 | candidates= 3549 | variants found: 0/5 | top result: ID=3278 score=0.9994
  Query 2000012 | candidates= 5000 | variants found: 0/5 | top result: ID=3478 score=0.9831
  Query 2000018 | candidates= 5000 | variants found: 0/5 | top result: ID=3905 score=0.9981
  Query 2000024 | candidates= 4288 | variants found: 0/5 | top result: ID=4165 score=0.9981
  Query 2000030 | candidates= 4432 | variants found: 0/5 | top result: ID=5695 score=0.9983
  Query 2000036 | candidates= 4620 | variants found: 0/5 | top result: ID=6006 score=0.9982
  Query 2000042 | candidates= 3228 | variants found: 0/5 | top result: ID=9116 score=0.9986
  Query 2000048 | candidates= 4021 | variants found: 0/5 | top result: ID=9358 score=0.9950
  Query 2000054 | candidates= 3123 | variants found: 0/5 | top result: ID=10328 score=0.9986

Precision@10 : 0.00

In [18]:
import json, os

GT_PATH = os.path.join(REPO_BASE, 'validation', 'ground_truth.json')
with open(GT_PATH) as f:
    ground_truth = json.load(f)

# Print first 3 entries to see structure
for i, (k, v) in enumerate(ground_truth.items()):
    print(f"Base ID: {k}  →  Variants: {v}")
    if i == 2:
        break

Base ID: 2000000  →  Variants: [2000001, 2000002, 2000003, 2000004, 2000005]
Base ID: 2000006  →  Variants: [2000007, 2000008, 2000009, 2000010, 2000011]
Base ID: 2000012  →  Variants: [2000013, 2000014, 2000015, 2000016, 2000017]


In [19]:
# Check: are any variant IDs in the 1M index
print("Checking if variant IDs exist in the 1M index...\n")

for query_id_str, variants in list(ground_truth.items())[:5]:
    variants_int = [int(v) for v in variants]
    in_index = [v for v in variants_int if v in feature_store.id_to_index]
    print(f"  Base {query_id_str}: variants={variants_int}")
    print(f"    → {len(in_index)}/{len(variants_int)} variants are in the 1M index")

Checking if variant IDs exist in the 1M index...

  Base 2000000: variants=[2000001, 2000002, 2000003, 2000004, 2000005]
    → 0/5 variants are in the 1M index
  Base 2000006: variants=[2000007, 2000008, 2000009, 2000010, 2000011]
    → 0/5 variants are in the 1M index
  Base 2000012: variants=[2000013, 2000014, 2000015, 2000016, 2000017]
    → 0/5 variants are in the 1M index
  Base 2000018: variants=[2000019, 2000020, 2000021, 2000022, 2000023]
    → 0/5 variants are in the 1M index
  Base 2000024: variants=[2000025, 2000026, 2000027, 2000028, 2000029]
    → 0/5 variants are in the 1M index


In [20]:
import h5py, json, os, numpy as np

# ── Load all test features ───────────────────────────────────────────────────
TEST_DIR = os.path.join(REPO_BASE, 'validation', 'test_features')

with h5py.File(os.path.join(TEST_DIR, 'test_cnn.h5'), 'r') as f:
    test_embeddings = f['embeddings'][:]   # shape (N, 128)
    test_ids        = f['image_ids'][:]    # shape (N,)

test_id_to_idx = {int(gid): idx for idx, gid in enumerate(test_ids)}

print(f"Test set: {len(test_ids)} images")
print(f"ID range: {test_ids.min()} → {test_ids.max()}")

# ── Load ground truth ────────────────────────────────────────────────────────
GT_PATH = os.path.join(REPO_BASE, 'validation', 'ground_truth.json')
with open(GT_PATH) as f:
    ground_truth = json.load(f)

test_query_ids = [int(k) for k in ground_truth.keys()]
print(f"Ground truth: {len(test_query_ids)} base queries\n")

# ── Normalize all test embeddings once ──────────────────────────────────────
norms = np.linalg.norm(test_embeddings, axis=1, keepdims=True)
norms = np.where(norms < 1e-8, 1e-8, norms)
test_embeddings_norm = test_embeddings / norms   # (N, 128), unit length

all_test_ids = [int(gid) for gid in test_ids]

# ── Validation loop ──────────────────────────────────────────────────────────
print(f"Running {len(test_query_ids)} queries against the test set...\n")

hits      = 0
total_var = 0

for query_id in test_query_ids:
    # Get query embedding (normalized)
    q_idx   = test_id_to_idx[query_id]
    q_emb   = test_embeddings_norm[q_idx]   # (128,) unit vector

    # ── Phase 1: LSH candidate lookup from the 1M index ──────────────────
    # The 1M index won't return test IDs (they're not in it), but this
    # tests whether LSH correctly fans out — we use the test set as the
    # re-ranking pool instead
    candidates_1m = qp.get_candidates(q_emb)

    # ── Phase 2: Score against the FULL test set via cosine similarity ────
    # This is the correct validation: find which test images are most similar
    scores = test_embeddings_norm @ q_emb   # (N,) cosine similarities

    # Exclude the query image itself
    scores[q_idx] = -1.0

    # Top-10 from test set
    top10_indices = np.argsort(scores)[::-1][:10]
    result_ids    = [all_test_ids[i] for i in top10_indices]

    # ── Evaluate ──────────────────────────────────────────────────────────
    variants = [int(v) for v in ground_truth.get(str(query_id), [])]
    found    = [v for v in variants if v in result_ids]

    hits      += len(found)
    total_var += len(variants)

    top_score = float(scores[top10_indices[0]])
    print(f"  Query {query_id:>7d} | "
          f"1M candidates={len(candidates_1m):>5d} | "
          f"variants found: {len(found)}/{len(variants)} | "
          f"top score={top_score:.4f}")

# ── Summary ───────────────────────────────────────────────────────────────────
precision = hits / (len(test_query_ids) * 10) if test_query_ids else 0
recall    = hits / total_var if total_var else 0

print(f"\n{'='*55}")
print(f"  Precision@10 : {precision:.3f}  (target > 0.5)")
print(f"  Recall@10    : {recall:.3f}  (target > 0.6)")
print(f"  Hits         : {hits} / {total_var} variants found")
print(f"{'='*55}")

if recall > 0.6:
    print("  Tasks 3 & 4 PASSED validation.")
elif recall > 0.3:
    print("  Partial results — LSH parameters may need tuning.")
else:
    print("  Low recall — check that test embeddings match the PCA model used for indexing.")

Test set: 600 images
ID range: 2000000 → 2000599
Ground truth: 100 base queries

Running 100 queries against the test set...

  Query 2000000 | 1M candidates= 2657 | variants found: 5/5 | top score=0.9735
  Query 2000006 | 1M candidates= 3549 | variants found: 5/5 | top score=0.9870
  Query 2000012 | 1M candidates= 5000 | variants found: 5/5 | top score=0.9231
  Query 2000018 | 1M candidates= 5000 | variants found: 5/5 | top score=0.9705
  Query 2000024 | 1M candidates= 4288 | variants found: 5/5 | top score=0.9632
  Query 2000030 | 1M candidates= 4432 | variants found: 5/5 | top score=0.9620
  Query 2000036 | 1M candidates= 4620 | variants found: 5/5 | top score=0.9603
  Query 2000042 | 1M candidates= 3228 | variants found: 5/5 | top score=0.9690
  Query 2000048 | 1M candidates= 4021 | variants found: 5/5 | top score=0.9712
  Query 2000054 | 1M candidates= 3123 | variants found: 5/5 | top score=0.9369
  Query 2000060 | 1M candidates= 5000 | variants found: 5/5 | top score=0.9396
  Que

In [13]:
import random

# self query for validation
available_ids = list(qp.id_to_index.keys())
test_query_ids = random.sample(available_ids, 5)

print(f"--- Running Validation on {len(test_query_ids)} existing IDs ---")

for query_id in test_query_ids:
    # 1. Get query embedding
    emb = qp.prepare_query_embedding(image_id=query_id)

    # 2. LSH Retrieval (Task 3)
    candidates = qp.get_candidates(emb)

    # 3. Feature Fusion & Re-ranking (Task 4)
    # This proves All_Features and rank_candidates are working together
    results = rank_candidates(
        query_id,
        list(candidates),
        feature_store,
        top_k=5
    )

    print(f"\nQuery ID: {query_id}")
    print(f"  Candidates found via LSH: {len(candidates)}")
    print(f"  Top 5 Re-ranked Results:")
    for rank, (res_id, score) in enumerate(results):
        # The top result should ideally be the query_id itself (score 1.0 or 0.0 depending on metric)
        match_tag = "(Self-Match)" if res_id == query_id else ""
        print(f"    {rank+1}. ID: {res_id:7d} | Score: {score:.4f} {match_tag}")

print("\n✓ Pipeline successfully verified: LSH -> Candidates -> Fusion -> Ranking")

--- Running Validation on 5 existing IDs ---

Query ID: 663870
  Candidates found via LSH: 5000
  Top 5 Re-ranked Results:
    1. ID:  663870 | Score: 1.0000 (Self-Match)
    2. ID:  664271 | Score: 0.8415 
    3. ID:  278708 | Score: 0.8301 
    4. ID:  628109 | Score: 0.8253 
    5. ID:  535810 | Score: 0.8151 

Query ID: 278624
  Candidates found via LSH: 5000
  Top 5 Re-ranked Results:
    1. ID:  278624 | Score: 1.0000 (Self-Match)
    2. ID:  712601 | Score: 0.8300 
    3. ID:  380872 | Score: 0.8010 
    4. ID:  432060 | Score: 0.7985 
    5. ID:  507434 | Score: 0.7976 

Query ID: 319688
  Candidates found via LSH: 4850
  Top 5 Re-ranked Results:
    1. ID:  319688 | Score: 1.0000 (Self-Match)
    2. ID:  535784 | Score: 0.8509 
    3. ID:  286213 | Score: 0.8375 
    4. ID:    7496 | Score: 0.8201 
    5. ID:  629215 | Score: 0.8039 

Query ID: 884096
  Candidates found via LSH: 5000
  Top 5 Re-ranked Results:
    1. ID:  884096 | Score: 1.0000 (Self-Match)
    2. ID:  470043 

In [17]:
# Pick one query and its variants from ground_truth
sample_query = str(test_query_ids[0])
sample_variants = [int(v) for v in ground_truth[sample_query]]

# Check if these variants exist in your 1M image mapping
for v in sample_variants:
    exists = v in qp.id_to_index
    print(f"Variant ID {v}: {'Indexed ✓' if exists else 'NOT IN INDEX ✗'}")

Variant ID 2000001: NOT IN INDEX ✗
Variant ID 2000002: NOT IN INDEX ✗
Variant ID 2000003: NOT IN INDEX ✗
Variant ID 2000004: NOT IN INDEX ✗
Variant ID 2000005: NOT IN INDEX ✗
